In [1]:
import os
import random
import shutil
import numpy as np
import pandas as pd
import tensorflow as tf
from keras.applications.vgg16 import VGG16, preprocess_input
from keras.preprocessing import image
from sklearn.cluster import KMeans
from time import time

In [2]:
# Root directory containing all embryo folders
data_root = '../images'

# Get a list of all image paths
all_image_paths = []

# Traverse through all embryo folders and collect image paths
for root, dirs, files in os.walk(data_root):
    if root.endswith('/F-30'):  # Only consider the last directory as 'F-30'
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                all_image_paths.append(os.path.join(root, file))

# Shuffle the list of image paths
random.shuffle(all_image_paths)

# Define the split ratio
train_ratio = 0.85
val_ratio = 0.15

# Calculate the split index
split_index = int(len(all_image_paths) * train_ratio)

# Split the image paths into training and validation sets
train_images = all_image_paths[:split_index]
val_images = all_image_paths[split_index:]

# Create directories for the training and validation sets
train_dir = os.path.join(data_root, 'train')
val_dir = os.path.join(data_root, 'val')

# If the training and validation directories exist, delete and recreate them
if os.path.exists(train_dir):
    shutil.rmtree(train_dir)
os.makedirs(train_dir)

if os.path.exists(val_dir):
    shutil.rmtree(val_dir)
os.makedirs(val_dir)

# Function to copy images to a destination directory
def copy_images(image_paths, dst_folder):
    for src_path in image_paths:
        dst_path = os.path.join(dst_folder, os.path.basename(src_path))
        try:
            if os.path.exists(dst_path):
                os.remove(dst_path)  # Remove the existing destination file
            shutil.copy(src_path, dst_path)
        except shutil.SameFileError:
            print(f"Skipping {src_path} because it is the same as {dst_path}")

# Copy the training set images to the training directory
copy_images(train_images, train_dir)

# Copy the validation set images to the validation directory
copy_images(val_images, val_dir)

print(f"Training set: {len(train_images)} images")
print(f"Validation set: {len(val_images)} images")


Training set: 3697 images
Validation set: 653 images


In [ ]:
# Load the pre-trained VGG16 model
base_model = VGG16(weights='imagenet', include_top=False)

def extract_features(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    
    # Extract features
    features = base_model.predict(x)
    # Apply average pooling
    features_mean = np.mean(features, axis=(1, 2))
    return features_mean

# Start timing
start_time = time()

# Extract features from the training and validation sets
train_features_list = []
val_features_list = []

for path in train_images:
    features = extract_features(path)
    train_features_list.append(features)

for path in val_images:
    features = extract_features(path)
    val_features_list.append(features)

# Convert feature lists to NumPy arrays
train_features_array = np.vstack(train_features_list)
val_features_array = np.vstack(val_features_list)

# Perform K-Means clustering
n_clusters = 2  # Number of clusters you want
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
kmeans.fit(train_features_array)

# Get the cluster labels for each sample
train_labels = kmeans.labels_
val_labels = kmeans.predict(val_features_array)

# Create a DataFrame for the training set labels
train_df = pd.DataFrame({
    'Image Name': [os.path.basename(path) for path in train_images],
    'Cluster Label': train_labels
})

# Create a DataFrame for the validation set labels
val_df = pd.DataFrame({
    'Image Name': [os.path.basename(path) for path in val_images],
    'Cluster Label': val_labels
})

output_dir = os.path.join(data_root, 'output.csv')

# Save the DataFrames to CSV files
train_df.to_csv(output_dir, index=False)
val_df.to_csv(output_dir, index=False)

# End timing
end_time = time()
elapsed_time = end_time - start_time

print(f"Cluster labels saved to CSV files.")
print(f"Total elapsed time: {elapsed_time:.2f} seconds")

1/1 [==============================] - 0s 87ms/step


/Users/lzz/anaconda3/envs/myenv/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


Training set cluster labels: [0 0 0 ... 1 0 0]
Validation set cluster labels: [0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 1
 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0
 0 0 1 0 0 0 0 1 0 0 0 0 0 1 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0
 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0
 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0
 0 1 0 1 1 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 1 0 1 1
 1 0 0 0 0 0 1 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0

In [ ]:
# Load age information
age_info = pd.read_csv('../path/to/age_info.csv')

# Merge training set with age information
train_df = pd.merge(train_df, age_info, on='Image Name', how='left')

# Merge validation set with age information
val_df = pd.merge(val_df, age_info, on='Image Name', how='left')

# Load embryo quality labels
labels = pd.read_csv('../path/to/embryo_quality_labels.csv')

# Merge training set with embryo quality labels
train_df = pd.merge(train_df, labels, on='Image Name', how='left')

# Merge validation set with embryo quality labels
val_df = pd.merge(val_df, labels, on='Image Name', how='left')

# Combine the training and validation sets into a single DataFrame
full_df = pd.concat([train_df, val_df], ignore_index=True)

# Re-split the data into training and validation sets
X = full_df[['Cluster Label', 'Age']].values
y = full_df['Embryo Quality'].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.15, random_state=42)

print(f"Training set: {len(X_train)} samples")
print(f"Validation set: {len(X_val)} samples")

# Initialize and train the logistic regression model
logreg = LogisticRegression(random_state=42)
logreg.fit(X_train, y_train)

# Predict on the validation set
y_pred = logreg.predict(X_val)

# Evaluate the model
accuracy = accuracy_score(y_val, y_pred)
report = classification_report(y_val, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:\n", report)

# Save the prediction results
val_df['Predicted Embryo Quality'] = y_pred

output_val_dir = os.path.join(output, 'prediction_LogReg_F0_val.csv')
val_df.to_csv(output_val_dir, index=False)

print(f"Prediction results saved to {output_val_dir}")

## CNN

In [93]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchsummary import summary
import torch.nn.functional as F

# CNN model
class ImprovedCNN(nn.Module):
    def __init__(self):
        super(ImprovedCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.fc1 = nn.Linear(128 * 64 * 64, 256)
        self.fc2 = nn.Linear(256, 1)  
        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)
        self.bn4 = nn.BatchNorm2d(256)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 128 * 64 * 64)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


    @staticmethod
    def weights_init(m):
        if isinstance(m, nn.Conv2d):
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)


transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.RandomHorizontalFlip(),  
    transforms.RandomVerticalFlip(),    
    transforms.RandomRotation(15),      
    transforms.ToTensor(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

# Load the training set and the verification set
train_dataset = datasets.ImageFolder(root='./data/flower_data/HW6training/HW6training/', transform=transform)
val_dataset = datasets.ImageFolder(root='./data/flower_data/HW6validation/HW6validation/', transform=transform)

# Create DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


model = ImprovedCNN()
criterion = nn.BCEWithLogitsLoss()  
optimizer = optim.SGD(model.parameters(), lr=0.0005, momentum=0.9)
num_epochs = 10


for epoch in range(num_epochs):
    # Train
    model.train()
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels.float().view(-1, 1))
        loss.backward()
        optimizer.step()

    # Valid
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []
    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model(images)
            loss = criterion(outputs, labels.float().view(-1, 1))
            val_loss += loss.item()
            total += labels.size(0)
            predicted = torch.round(torch.sigmoid(outputs))
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.view(-1, 1).cpu().numpy())
            correct += (predicted == labels.view(-1, 1)).sum().item()

    val_accuracy = correct / total
    average_val_loss = val_loss / len(val_loader)

    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}, Validation Loss: {average_val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}')

conf_matrix = confusion_matrix(all_labels, all_predictions)
print("Confusion Matrix:")
print(conf_matrix)

# Summary
summary(model, (3, 512, 512))


Epoch [1/10], Loss: 0.6334, Validation Loss: 0.6768, Validation Accuracy: 0.5732
Epoch [2/10], Loss: 0.6512, Validation Loss: 0.6781, Validation Accuracy: 0.5610
Epoch [3/10], Loss: 0.6485, Validation Loss: 0.6786, Validation Accuracy: 0.5122
Epoch [4/10], Loss: 0.6540, Validation Loss: 0.6707, Validation Accuracy: 0.5610
Epoch [5/10], Loss: 0.6166, Validation Loss: 0.6623, Validation Accuracy: 0.5366
Epoch [6/10], Loss: 0.5724, Validation Loss: 0.6401, Validation Accuracy: 0.5244
Epoch [7/10], Loss: 0.7690, Validation Loss: 0.6914, Validation Accuracy: 0.6951
Epoch [8/10], Loss: 0.7702, Validation Loss: 0.6719, Validation Accuracy: 0.7195
Epoch [9/10], Loss: 0.8089, Validation Loss: 0.6919, Validation Accuracy: 0.6951
Epoch [10/10], Loss: 0.8458, Validation Loss: 0.6891, Validation Accuracy: 0.7439
Confusion Matrix:
[[34  1]
 [20 27]]
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1   

In [94]:
torch.save(model.state_dict(), 'simple_cnn_model.pth')

In [96]:
import torch
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision import datasets
import pandas as pd

# Define the image preprocessing transformation
transform = transforms.Compose([
    transforms.Resize((512, 512)),  # Resize the image to 512x512 pixels
    transforms.RandomHorizontalFlip(),  # Randomly flip the image horizontally
    transforms.RandomVerticalFlip(),    # Randomly flip the image vertically
    transforms.RandomRotation(15),      # Randomly rotate the image by up to 15 degrees
    transforms.ToTensor(),              # Convert the image to a PyTorch Tensor
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  # Randomly adjust brightness and contrast
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize the image with mean and standard deviation
])

# Load the unknown testing dataset
unknown_dataset = datasets.ImageFolder(root='./HW6testing/', transform=transform)  
unknown_loader = DataLoader(unknown_dataset, batch_size=32, shuffle=False)

# Initialize the model
model = ImprovedCNN()
model.load_state_dict(torch.load('./simple_cnn_model.pth'))  # Replace 'path/to/your/trained/model.pth' with the actual path
model.eval()

# Make predictions on the unknown testing dataset
predictions = []
with torch.no_grad():
    for images, _ in unknown_loader:
        outputs = model(images)
        predicted_labels = torch.round(torch.sigmoid(outputs)).cpu().numpy().astype(int)
        predictions.extend(predicted_labels.flatten())

# Create a DataFrame with filenames and corresponding predictions
filenames = [f"UK_{i+1}" for i in range(len(predictions))]
df = pd.DataFrame(list(zip(filenames, predictions)), columns=['Filename', 'Prediction'])

# Save the DataFrame to a CSV file
df.to_csv('predictions_CNN.csv', index=False)

## MobileNetV2

In [35]:
import os
import sys
import json

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from tqdm import tqdm
from sklearn.metrics import confusion_matrix
from model_v2 import MobileNetV2


def main():
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print("using {} device.".format(device))

    batch_size = 32
    epochs = 20

    data_transform = {
        "train": transforms.Compose([transforms.RandomResizedCrop(224),
                                     transforms.RandomHorizontalFlip(),
                                     transforms.ToTensor(),
                                     transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
        "val": transforms.Compose([transforms.Resize(256),
                                   transforms.CenterCrop(224),
                                   transforms.ToTensor(),
                                   transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])}

    data_root = os.path.abspath(os.path.join(os.getcwd(),"."))  # get data root path
    image_path = os.path.join(data_root, "data", "flower_data")  # flower data set path
    assert os.path.exists(image_path), "{} path does not exist.".format(image_path)
    train_dataset = datasets.ImageFolder(root=os.path.join(image_path, "HW6training/HW6training"),
                                         transform=data_transform["train"])
    train_num = len(train_dataset)

    flower_list = train_dataset.class_to_idx
    cla_dict = dict((val, key) for key, val in flower_list.items())
    # write dict into json file
    json_str = json.dumps(cla_dict, indent=4)
    with open('class_indices.json', 'w') as json_file:
        json_file.write(json_str)

    nw = min([os.cpu_count(), batch_size if batch_size > 1 else 0, 8])  # number of workers
    print('Using {} dataloader workers every process'.format(nw))

    train_loader = torch.utils.data.DataLoader(train_dataset,
                                               batch_size=batch_size, shuffle=True,
                                               num_workers=nw)

    validate_dataset = datasets.ImageFolder(root=os.path.join(image_path, "HW6validation/HW6validation"),
                                            transform=data_transform["val"])
    val_num = len(validate_dataset)
    validate_loader = torch.utils.data.DataLoader(validate_dataset,
                                                  batch_size=batch_size, shuffle=False,
                                                  num_workers=nw)

    print("using {} images for training, {} images for validation.".format(train_num,
                                                                           val_num))

    # create model
    net = MobileNetV2(num_classes=2)

    # load pretrain weights
    # download url: https://download.pytorch.org/models/mobilenet_v2-b0353104.pth
    model_weight_path = "./mobilenet_v2.pth"
    assert os.path.exists(model_weight_path), "file {} dose not exist.".format(model_weight_path)
    pre_weights = torch.load(model_weight_path, map_location=device)

    # delete classifier weights
    pre_dict = {k: v for k, v in pre_weights.items() if net.state_dict()[k].numel() == v.numel()}
    missing_keys, unexpected_keys = net.load_state_dict(pre_dict, strict=False)

    # freeze features weights
    for param in net.features.parameters():
        param.requires_grad = False

    net.to(device)

    # define loss function
    loss_function = nn.CrossEntropyLoss()

    # construct an optimizer
    params = [p for p in net.parameters() if p.requires_grad]
    optimizer = optim.Adam(params, lr=0.0001)

    best_acc = 0.0
    save_path = './MobileNetV2_test.pth'
    train_steps = len(train_loader)
    for epoch in range(epochs):
        # train
        net.train()
        running_loss = 0.0
        train_bar = tqdm(train_loader, file=sys.stdout)
        for step, data in enumerate(train_bar):
            images, labels = data
            optimizer.zero_grad()
            logits = net(images.to(device))
            loss = loss_function(logits, labels.to(device))
            loss.backward()
            optimizer.step()

            # print statistics
            running_loss += loss.item()

            train_bar.desc = "train epoch[{}/{}] loss:{:.3f}".format(epoch + 1,
                                                                     epochs,
                                                                     loss)

        # validate
        net.eval()
        acc = 0.0  # accumulate accurate number / epoch
        predictions = []
        true_labels = []
        with torch.no_grad():
            val_bar = tqdm(validate_loader, file=sys.stdout)
            for val_data in val_bar:
                val_images, val_labels = val_data
                outputs = net(val_images.to(device))
                # loss = loss_function(outputs, test_labels)
                predict_y = torch.max(outputs, dim=1)[1]
                acc += torch.eq(predict_y, val_labels.to(device)).sum().item()

                predictions.extend(predict_y.cpu().numpy())
                true_labels.extend(val_labels.cpu().numpy())
                
                val_bar.desc = "valid epoch[{}/{}]".format(epoch + 1,
                                                           epochs)
        val_accurate = acc / val_num
        print('[epoch %d] train_loss: %.3f  val_accuracy: %.3f' %
              (epoch + 1, running_loss / train_steps, val_accurate))

        if val_accurate > best_acc:
            best_acc = val_accurate
            torch.save(net.state_dict(), save_path)

    print('Finished Training')
    # Calculate and print confusion matrix
    conf_matrix = confusion_matrix(true_labels, predictions)
    print("Confusion Matrix:")
    print(conf_matrix)

if __name__ == '__main__':
    main()


using cpu device.
Using 8 dataloader workers every process
using 690 images for training, 82 images for validation.
valid epoch[1/20]: 100%|██████████| 3/3 [00:14<00:00,  4.85s/it]
[epoch 1] train_loss: 0.679  val_accuracy: 0.561
valid epoch[2/20]: 100%|██████████| 3/3 [00:09<00:00,  3.30s/it]
[epoch 2] train_loss: 0.636  val_accuracy: 0.683
valid epoch[3/20]: 100%|██████████| 3/3 [00:14<00:00,  4.85s/it]
[epoch 3] train_loss: 0.599  val_accuracy: 0.707
valid epoch[4/20]: 100%|██████████| 3/3 [00:14<00:00,  4.81s/it]
[epoch 4] train_loss: 0.582  val_accuracy: 0.707
valid epoch[5/20]: 100%|██████████| 3/3 [00:10<00:00,  3.59s/it]
[epoch 5] train_loss: 0.550  val_accuracy: 0.671
valid epoch[6/20]: 100%|██████████| 3/3 [00:09<00:00,  3.32s/it]
[epoch 6] train_loss: 0.535  val_accuracy: 0.683
valid epoch[7/20]: 100%|██████████| 3/3 [00:14<00:00,  4.96s/it]
[epoch 7] train_loss: 0.528  val_accuracy: 0.732
valid epoch[8/20]: 100%|██████████| 3/3 [00:10<00:00,  3.43s/it]
[epoch 8] train_loss:

In [68]:
import os
import json
import numpy as np
import torch
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt
import os
import csv
from model_v2 import MobileNetV2


def main():
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    file_csv="./1.csv"
    f=open(file_csv, 'w', newline="", encoding='utf-8')
    wf = csv.writer(f)
    wf.writerow(['Filename', 'Prediction'])
    data_transform = transforms.Compose(
        [transforms.Resize(256),
         transforms.CenterCrop(224),
         transforms.ToTensor(),
         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    test_file="./HW6testing/HW6testing/"
    files_list = os.listdir(test_file)
    print(files_list)
    for img_name in files_list:
        img_path=os.path.join(test_file,img_name)
        print(img_path)
    # load image
        #img_path = "../tulip.jpg"
        assert os.path.exists(img_path), "file: '{}' dose not exist.".format(img_path)
        img = Image.open(img_path)
        #plt.imshow(img)
        # [N, C, H, W]
        img = data_transform(img)
        # expand batch dimension
        img = torch.unsqueeze(img, dim=0)

        # read class_indict
        json_path = './class_indices.json'
        assert os.path.exists(json_path), "file: '{}' dose not exist.".format(json_path)

        json_file = open(json_path, "r")
        class_indict = json.load(json_file)

        # create model
        model = MobileNetV2(num_classes=2).to(device)
        # load model weights
        model_weight_path = "./MobileNetV2.pth"
        model.load_state_dict(torch.load(model_weight_path, map_location=device))
        model.eval()
        with torch.no_grad():
            # predict class
            output = torch.squeeze(model(img.to(device))).cpu()
            predict = torch.softmax(output, dim=0)
            predict_cla = torch.argmax(predict).numpy()

        #print(predict.numpy())
        #print(np.argmax(predict.numpy()))
        i=int(np.argmax(predict.numpy()))
        #print(img_name)
        #print(i)
        data=[img_name,i]
        wf.writerow(data)
        #f.write(str(np.argmax(predict.numpy())))
        #f.write(',')
        #print_res = "class: {}   prob: {:.3}".format(class_indict[str(predict_cla)],predict[predict_cla].numpy())
        #plt.title(print_res)
        #for i in range(len(predict)):
        #    print("class: {:10}   prob: {:.3}".format(class_indict[str(i)],predict[i].numpy()))
        #plt.show()
    f.close()

if __name__ == '__main__':
    main()


['UK_1.png', 'UK_10.png', 'UK_100.png', 'UK_101.png', 'UK_102.png', 'UK_103.png', 'UK_104.png', 'UK_105.png', 'UK_106.png', 'UK_107.png', 'UK_108.png', 'UK_109.png', 'UK_11.png', 'UK_110.png', 'UK_111.png', 'UK_112.png', 'UK_113.png', 'UK_114.png', 'UK_115.png', 'UK_116.png', 'UK_117.png', 'UK_118.png', 'UK_119.png', 'UK_12.png', 'UK_120.png', 'UK_121.png', 'UK_122.png', 'UK_123.png', 'UK_124.png', 'UK_125.png', 'UK_126.png', 'UK_127.png', 'UK_128.png', 'UK_129.png', 'UK_13.png', 'UK_130.png', 'UK_131.png', 'UK_132.png', 'UK_133.png', 'UK_134.png', 'UK_135.png', 'UK_137.png', 'UK_138.png', 'UK_139.png', 'UK_14.png', 'UK_140.png', 'UK_141.png', 'UK_143.png', 'UK_144.png', 'UK_145.png', 'UK_146.png', 'UK_148.png', 'UK_149.png', 'UK_15.png', 'UK_150.png', 'UK_151.png', 'UK_152.png', 'UK_153.png', 'UK_154.png', 'UK_157.png', 'UK_159.png', 'UK_16.png', 'UK_160.png', 'UK_161.png', 'UK_162.png', 'UK_17.png', 'UK_18.png', 'UK_19.png', 'UK_2.png', 'UK_20.png', 'UK_21.png', 'UK_22.png', 'UK_23.p

In [69]:
import re

# Define the function to extract numbers from filenames
def extract_number(filename):
    match = re.search(r'\d+', filename)
    if match:
        return int(match.group())
    return 0  # If no number is found, default to 0

# Read the CSV file
csv_file = "1.1.csv"  # Please replace with your actual CSV file path
with open(csv_file, 'r') as f:
    reader = csv.reader(f)
    header = next(reader)  # Read the header
    rows = sorted(reader, key=lambda x: extract_number(x[0]))  # Sort by the extracted number

# Write the sorted data to a new CSV file
sorted_csv_file = "sorted_" + csv_file
with open(sorted_csv_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(header)  # Write the header
    writer.writerows(rows)  # Write the sorted rows

## ALEXNET

In [86]:
import os
import sys
import json

import torch
import torch.nn as nn
from torchvision import transforms, datasets, utils
import matplotlib.pyplot as plt
import numpy as np
import torch.optim as optim
from tqdm import tqdm

from model import AlexNet


def main():
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print("using {} device.".format(device))

    data_transform = {
        "train": transforms.Compose([transforms.RandomResizedCrop(224),
                                     transforms.RandomHorizontalFlip(),
                                     transforms.ToTensor(),
                                     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]),
        "val": transforms.Compose([transforms.Resize((224, 224)),  # cannot 224, must (224, 224)
                                   transforms.ToTensor(),
                                   transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])}

    data_root = os.path.abspath(os.path.join(os.getcwd(), "."))  # get data root path
    image_path = os.path.join(data_root, "data", "flower_data")  # flower data set path
    assert os.path.exists(image_path), "{} path does not exist.".format(image_path)
    train_dataset = datasets.ImageFolder(root=os.path.join(image_path, "HW6training/HW6training"),
                                         transform=data_transform["train"])
    train_num = len(train_dataset)

    # {'daisy':0, 'dandelion':1, 'roses':2, 'sunflower':3, 'tulips':4}
    flower_list = train_dataset.class_to_idx
    cla_dict = dict((val, key) for key, val in flower_list.items())
    # write dict into json file
    json_str = json.dumps(cla_dict, indent=4)
    with open('class_indices.json', 'w') as json_file:
        json_file.write(json_str)

    batch_size = 32
    nw = min([os.cpu_count(), batch_size if batch_size > 1 else 0, 8])  # number of workers
    print('Using {} dataloader workers every process'.format(nw))

    train_loader = torch.utils.data.DataLoader(train_dataset,
                                               batch_size=batch_size, shuffle=True,
                                               num_workers=nw)

    validate_dataset = datasets.ImageFolder(root=os.path.join(image_path, "HW6validation/HW6validation"),
                                            transform=data_transform["val"])
    val_num = len(validate_dataset)
    validate_loader = torch.utils.data.DataLoader(validate_dataset,
                                                  batch_size=4, shuffle=False,
                                                  num_workers=nw)

    print("using {} images for training, {} images for validation.".format(train_num,
                                                                           val_num))
    # test_data_iter = iter(validate_loader)
    # test_image, test_label = test_data_iter.next()
    #
    # def imshow(img):
    #     img = img / 2 + 0.5  # unnormalize
    #     npimg = img.numpy()
    #     plt.imshow(np.transpose(npimg, (1, 2, 0)))
    #     plt.show()
    #
    # print(' '.join('%5s' % cla_dict[test_label[j].item()] for j in range(4)))
    # imshow(utils.make_grid(test_image))

    net = AlexNet(num_classes=2, init_weights=True)

    net.to(device)
    loss_function = nn.CrossEntropyLoss()
    # pata = list(net.parameters())
    optimizer = optim.Adam(net.parameters(), lr=0.0002)

    epochs = 20
    save_path = './AlexNet.pth'
    best_acc = 0.0
    train_steps = len(train_loader)
    for epoch in range(epochs):
        # train
        net.train()
        running_loss = 0.0
        train_bar = tqdm(train_loader, file=sys.stdout)
        for step, data in enumerate(train_bar):
            images, labels = data
            optimizer.zero_grad()
            outputs = net(images.to(device))
            loss = loss_function(outputs, labels.to(device))
            loss.backward()
            optimizer.step()

            # print statistics
            running_loss += loss.item()

            train_bar.desc = "train epoch[{}/{}] loss:{:.3f}".format(epoch + 1,
                                                                     epochs,
                                                                     loss)

        # validate
        net.eval()
        acc = 0.0  # accumulate accurate number / epoch
        with torch.no_grad():
            val_bar = tqdm(validate_loader, file=sys.stdout)
            for val_data in val_bar:
                val_images, val_labels = val_data
                outputs = net(val_images.to(device))
                predict_y = torch.max(outputs, dim=1)[1]
                acc += torch.eq(predict_y, val_labels.to(device)).sum().item()

        val_accurate = acc / val_num
        print('[epoch %d] train_loss: %.3f  val_accuracy: %.3f' %
              (epoch + 1, running_loss / train_steps, val_accurate))

        if val_accurate > best_acc:
            best_acc = val_accurate
            torch.save(net.state_dict(), save_path)

    print('Finished Training')


if __name__ == '__main__':
    main()


using cpu device.
Using 8 dataloader workers every process
using 690 images for training, 82 images for validation.
100%|██████████| 21/21 [00:07<00:00,  2.83it/s]
[epoch 1] train_loss: 0.690  val_accuracy: 0.427
100%|██████████| 21/21 [00:12<00:00,  1.64it/s]
[epoch 2] train_loss: 0.621  val_accuracy: 0.646
100%|██████████| 21/21 [00:08<00:00,  2.39it/s]
[epoch 3] train_loss: 0.595  val_accuracy: 0.634
100%|██████████| 21/21 [00:11<00:00,  1.84it/s]
[epoch 4] train_loss: 0.564  val_accuracy: 0.671
100%|██████████| 21/21 [00:08<00:00,  2.33it/s]
[epoch 5] train_loss: 0.571  val_accuracy: 0.732
100%|██████████| 21/21 [00:09<00:00,  2.19it/s]
[epoch 6] train_loss: 0.569  val_accuracy: 0.634
100%|██████████| 21/21 [00:14<00:00,  1.48it/s]
[epoch 7] train_loss: 0.525  val_accuracy: 0.646
100%|██████████| 21/21 [00:14<00:00,  1.49it/s]
[epoch 8] train_loss: 0.573  val_accuracy: 0.659
100%|██████████| 21/21 [00:14<00:00,  1.48it/s]
[epoch 9] train_loss: 0.539  val_accuracy: 0.598
100%|██████

In [87]:
import os
import json

import torch
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt

from model import AlexNet


def main():
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    file_csv="./2.csv"
    f=open(file_csv, 'w', newline="", encoding='utf-8')
    wf = csv.writer(f)
    wf.writerow(['Filename', 'Prediction'])
    data_transform = transforms.Compose(
        [transforms.Resize((224, 224)),
         transforms.ToTensor(),
         transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
    test_file="./HW6testing/HW6testing/"
    files_list = os.listdir(test_file)
    print(files_list)
    for img_name in files_list:
        img_path=os.path.join(test_file,img_name)
        print(img_path)
        
        # load image
        assert os.path.exists(img_path), "file: '{}' dose not exist.".format(img_path)
        img = Image.open(img_path)

        # [N, C, H, W]
        img = data_transform(img)
        # expand batch dimension
        img = torch.unsqueeze(img, dim=0)

        # read class_indict
        json_path = './class_indices.json'
        assert os.path.exists(json_path), "file: '{}' dose not exist.".format(json_path)

        json_file = open(json_path, "r")
        class_indict = json.load(json_file)

        # create model
        model = AlexNet(num_classes=2).to(device)

        # load model weights
        weights_path = "./AlexNet.pth"
        assert os.path.exists(weights_path), "file: '{}' dose not exist.".format(weights_path)
        model.load_state_dict(torch.load(weights_path))

        model.eval()
        with torch.no_grad():
            # predict class
            output = torch.squeeze(model(img.to(device))).cpu()
            predict = torch.softmax(output, dim=0)
            predict_cla = torch.argmax(predict).numpy()

        #print(predict.numpy())
        #print(np.argmax(predict.numpy()))
        i=int(np.argmax(predict.numpy()))
        #print(img_name)
        #print(i)
        data=[img_name,i]
        wf.writerow(data)
        #f.write(str(np.argmax(predict.numpy())))
        #f.write(',')
        #print_res = "class: {}   prob: {:.3}".format(class_indict[str(predict_cla)],predict[predict_cla].numpy())
        #plt.title(print_res)
        #for i in range(len(predict)):
        #    print("class: {:10}   prob: {:.3}".format(class_indict[str(i)],predict[i].numpy()))
        #plt.show()
    f.close()


if __name__ == '__main__':
    main()


['UK_1.png', 'UK_10.png', 'UK_100.png', 'UK_101.png', 'UK_102.png', 'UK_103.png', 'UK_104.png', 'UK_105.png', 'UK_106.png', 'UK_107.png', 'UK_108.png', 'UK_109.png', 'UK_11.png', 'UK_110.png', 'UK_111.png', 'UK_112.png', 'UK_113.png', 'UK_114.png', 'UK_115.png', 'UK_116.png', 'UK_117.png', 'UK_118.png', 'UK_119.png', 'UK_12.png', 'UK_120.png', 'UK_121.png', 'UK_122.png', 'UK_123.png', 'UK_124.png', 'UK_125.png', 'UK_126.png', 'UK_127.png', 'UK_128.png', 'UK_129.png', 'UK_13.png', 'UK_130.png', 'UK_131.png', 'UK_132.png', 'UK_133.png', 'UK_134.png', 'UK_135.png', 'UK_137.png', 'UK_138.png', 'UK_139.png', 'UK_14.png', 'UK_140.png', 'UK_141.png', 'UK_143.png', 'UK_144.png', 'UK_145.png', 'UK_146.png', 'UK_148.png', 'UK_149.png', 'UK_15.png', 'UK_150.png', 'UK_151.png', 'UK_152.png', 'UK_153.png', 'UK_154.png', 'UK_157.png', 'UK_159.png', 'UK_16.png', 'UK_160.png', 'UK_161.png', 'UK_162.png', 'UK_17.png', 'UK_18.png', 'UK_19.png', 'UK_2.png', 'UK_20.png', 'UK_21.png', 'UK_22.png', 'UK_23.p

In [88]:
import re
import csv

# Define the function to extract numbers from a string
def extract_number(filename):
    match = re.search(r'\d+', filename)
    if match:
        return int(match.group())
    return 0  # If no number is found, default to 0

# Read the CSV file
csv_file = "2.csv"  # Please replace with your actual CSV file path
with open(csv_file, 'r') as f:
    reader = csv.reader(f)
    header = next(reader)  # Read the header
    rows = sorted(reader, key=lambda x: extract_number(x[0]))  # Sort by the extracted number

# Write the sorted data to a new CSV file
sorted_csv_file = "sorted_" + csv_file
with open(sorted_csv_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(header)  # Write the header
    writer.writerows(rows)  # Write the sorted rows

In [25]:
import sqlite3
import pandas as pd


def saveImage(_id, byte):
    f = open('%s_image.png' % _id, 'wb')
    f.write(byte)
    f.close()
    return


if __name__ == '__main__':
    conn = sqlite3.connect("/Users/lzz/Documents/GitHub/IVF_AI/database/image.pdb")
    cur = conn.cursor()
    cur.execute("select * from IMAGES")

    # 获取table所有字段名
    col_name_list = [tuple[0] for tuple in cur.description]

    # 所有记录列表
    record_all = cur.fetchall()
    df = pd.DataFrame(record_all, columns=col_name_list)

    for index in range(len(df)):
        bytes_id = df['Well'][index]
        bytes_img = df['Image'][index]
        saveImage(bytes_id, bytes_img)

    print("----------------执行完成-----总个数：%s" % len(df))

----------------执行完成-----总个数：27588


Failed to process image: image file is truncated (1 bytes not processed)
Base64 string is not valid image data.
